# QA Answer Generation Pipeline

This notebook generates answers for previously generated questions using Gemini API batch processing.

**Prerequisites:** Questions must be generated first using `[QA]-batch-generation.ipynb`

---

## Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Configuration](#2-configuration)
3. [Load Generated Questions](#3-load-generated-questions)
4. [Batch Processing Utilities](#4-batch-processing-utilities)
5. [Answer Generation](#5-answer-generation)
6. [Save Results](#6-save-results)

---

In [ ]:
# %%capture
# %pip install -q -U google-genai
# %pip install google-cloud-aiplatform

## 1. Setup & Installation

In [ ]:
from google import genai
import os
import dotenv
import pandas as pd
from google.genai import types
import json
from typing import List, Dict, Callable, Optional, Any
import numpy as np
import time

## 2. Configuration

Includes:
- **Paths**: Input/output file paths
- **Model Settings**: API key, model name, batch size
- **Generation Config**: A_GEN_CONFIG
- **Prompt**: A_INSTRUCTION

In [ ]:
import os
# ==================== CONFIGURATION ====================
# Local Paths
# INPUT_Q_PATH = "../data/raw/Q_data.parquet"  # Generated questions file
# TEMP_BATCH_FOLDER = "./temp/batches/"
# OUTPUT_QA_PARQUET = "./temp/QA_data.parquet"
# CACHE_SAVE_FOLDER = "./temp/cache/"

# Kaggle Paths
CACHE_SAVE_FOLDER = "../data/cache/"
CACHE_LOAD_FOLDER = "../data/cache/"
TEMP_BATCH_FOLDER = "../data/cache/batches/"
INPUT_Q_PATH = "../data/processed/Q_data.parquet" # part 2 and 3 (the rest)
OUTPUT_QA_PARQUET = "../data/processed/QA_data_p2.parquet"

BATCH_A_PATH = os.path.join(TEMP_BATCH_FOLDER, "batch_answer_requests_{}.jsonl")
os.makedirs(CACHE_SAVE_FOLDER, exist_ok=True)
os.makedirs(TEMP_BATCH_FOLDER, exist_ok=True)

part_index = 0  # Which part to process (if questions were split into parts)

# Resume configuration
RESUME_FROM_LAST_CHECKPOINT = True

# Model settings
MODEL_NAME = "gemini-2.5-flash"
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") # My new API key


# Batch processing
BATCH_CHUNK_SIZE = 2000

# Generation config
A_GEN_CONFIG = {
    "max_output_tokens": 200,
    "temperature": 0.7,
    "top_p": 0.9,
    "thinking_config": {"thinking_budget": 0}
}

# ==================== PROMPT ====================
A_INSTRUCTION = """You are an expert at answering financial questions based on provided context.

## CONTEXT INFORMATION
**Article Title:** {title}
**Publication Date:** {timestamp}

**Content:**
{text}

## QUESTION TO ANSWER
{question}

## ANSWER RULES (MANDATORY)
1. **ONLY use information from the context above** - STRICTLY FORBIDDEN to add external knowledge
2. Answer DIRECTLY, CONCISELY, ACCURATELY
3. Use numbers and proper names EXACTLY as in the text
4. If question cannot be answered from context → return: "Thông tin không có trong văn bản"
5. Length: 1-3 sentences (maximum 50 words)
6. **ALL answers MUST be written in Vietnamese**

## GUIDANCE BY QUESTION TYPE

### Factoid Extraction
- Answer directly with specific numbers/information
- Example: "15.234 tỷ VNĐ, tăng 18% so với cùng kỳ năm trước"

### Summary & Interpretation
- Summarize main points mentioned
- Example: "Do tăng trưởng tín dụng mạnh và kiểm soát chi phí tốt"

### Comparison
- State comparison result with specific numbers
- Example: "Tăng 2.500 tỷ VNĐ, tương đương mức tăng trưởng 12% so với quý trước"

### Verification/Confirmation
- Answer Yes/No + brief explanation (if needed)
- Example: "Có, công ty đã chi trả cổ tức ở mức 15%"

## NOTES
- Do not add personal opinions or analysis beyond the text
- Do not start with "Theo văn bản...", "Dựa trên nội dung..." (redundant)
- Prioritize citing accurate numbers from the source
- If multiple relevant information exists, synthesize into 1-2 coherent sentences
- **Remember: Write ALL answers in Vietnamese**

Now answer the question:"""

# ==================== CLIENT ====================nhìn
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
from tqdm import tqdm

pending_batches = client.batches.list(config={'page_size':100000})
num_pending_batches = len(pending_batches)
print(f"Found {num_pending_batches} batches in execution state")
num_batch_deleted = 0

for job in tqdm(pending_batches):
    try:
        client.batches.cancel(name=job.name)
        client.batches.delete(name=job.name)
        num_batch_deleted += 1
    except Exception as e:
        print(f"Skipping {job.name} due to error: {e}")

if num_batch_deleted > 0:
    print(f"Cleanup complete. {num_batch_deleted} batches removed")
    time.sleep(180) # Wait 3 minutes until proceed

## 3. Load Generated Questions

Load previously generated questions from parquet file.

In [ ]:
# Load saved questions
df = pd.read_parquet(INPUT_Q_PATH, engine="pyarrow")

# Optional: Split into parts if needed
parts = np.array_split(df, 2)
df_questions = parts[part_index]

# df_questions = df  # Process all questions

print(f"[INFO] Loaded questions from: {INPUT_Q_PATH}")
print(f"[INFO] Total chunks: {len(df_questions)}")
print(f"[INFO] Total questions: {sum(len(q) for q in df_questions['questions'])}")

# Reconstruct q_generation_results from saved data
q_generation_results = {}
for _, row in df_questions.iterrows():
    key = str(row['input_idx'])
    q_generation_results[key] = row['questions']

# Reconstruct inputs from saved data
inputs = []
for _, row in df_questions.iterrows():
    inputs.append({
        'index': row['article_idx'],
        'chunk_idx': row['chunk_idx'],
        'url': row['url'],
        'title': row['title'],
        'time': row['time'],
        'text': row['context']
    })

print(f"[INFO] Reconstructed {len(inputs)} inputs and {len(q_generation_results)} question results")

## 4. Batch Processing Utilities

Reusable functions for batch file creation, job submission, and result parsing.

In [ ]:
# ==================== BATCH PROCESSING UTILITIES ====================

def upload_and_submit_batch(filename: str) -> Any:
    """Upload a file and create a batch job."""
    print(f"[INFO] Uploading {filename}...")
    uploaded = client.files.upload(
        file=filename,
        config=types.UploadFileConfig(
            display_name="generation-requests",
            mime_type="text/plain"
        )
    )
    print(f"[INFO] Creating batch job for {uploaded.name}...")
    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded.name,
        config={"display_name": "generation-run"}
    )
    print(f"[INFO] Job created: {batch_job.name}")
    return batch_job

def parse_batch_results(
    records_list: List[List[Dict]],
    parser: Optional[Callable[[str], Any]] = None
):
    """Parse batch results with optional custom parser."""
    results = {}
    errors = []
    
    for records in records_list:
        for rec in records:
            try:
                key = rec.get("key", "")
                result_text = rec["response"]["candidates"][0]["content"]["parts"][0]["text"]
                if parser:
                    results[key] = parser(result_text)
                else:
                    results[key] = result_text.strip()
            except Exception as e:
                errors.append({"record": rec, "error": str(e)})
    
    print(f"[INFO] Successfully parsed: {len(results)} results")
    print(f"[INFO] Errors: {len(errors)}")
    return results, errors

def wait_for_job_completion(job_name: str, poll_interval: int = 5) -> Any:
    """Poll for batch job completion with progress updates."""
    print(f"[INFO] Waiting for job {job_name}...")
    start_time = time.time()
    
    while True:
        job = client.batches.get(name=job_name)
        state = job.state.name
        
        if state in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"):
            break
        time.sleep(poll_interval)
    
    elapsed = (time.time() - start_time) / 60
    if state == "JOB_STATE_SUCCEEDED":
        print(f"[SUCCESS] Job {job_name} completed in {elapsed:.1f} min")
    else:
        print(f"[ERROR] Job {job_name} ended with state: {state} after {elapsed:.1f} min")
    return job

def process_single_batch(
    filename: str,
    job_name: str,
    parser: Optional[Callable[[str], Any]] = None
) -> tuple[Dict, List[Dict]]:
    """
    Download, parse, and return results from a single completed batch job.
    Returns: (results_dict, errors_list)
    """
    try:
        print(f"[INFO] Downloading results for {filename} (job: {job_name})...")
        job = client.batches.get(name=job_name)
        result_file = job.dest.file_name
        raw = client.files.download(file=result_file).decode('utf-8')
        records = [json.loads(line) for line in raw.splitlines() if line.strip()]
        
        print(f"[INFO] Downloaded {len(records)} records from {filename}")
        
        # Use the existing parse_batch_results function
        results, errors = parse_batch_results([records], parser=parser)
        
        return results, errors
        
    except Exception as e:
        print(f"[ERROR] Failed to download/parse {filename}: {e}")
        return {}, []


def create_batch_files(
    inputs: List[Dict],
    batch_path_template: str,
    prompt_formatter: Callable[[Dict], str],
    gen_config: Dict,
    chunk_size: int = BATCH_CHUNK_SIZE
) -> List[str]:
    """Create JSONL batch files from inputs."""
    filenames = []
    file_idx = 1
    
    for i in range(0, len(inputs), chunk_size):
        batch = inputs[i:i + chunk_size]
        filename = batch_path_template.format(file_idx)
        file_idx += 1
        
        with open(filename, "w", encoding='utf-8') as f:
            for batch_idx, item in enumerate(batch):
                global_idx = i + batch_idx
                record = {
                    "key": str(global_idx),
                    "request": {
                        "contents": [{"parts": [{"text": prompt_formatter(item)}]}],
                        "generation_config": gen_config,
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        filenames.append(filename)
    
    print(f"[INFO] Created {len(filenames)} batch files ({len(inputs)} total items)")
    return filenames

## 5. Answer Generation

Generate answers for all questions using incremental batch processing.

In [ ]:
def parse_answer(result_text: str) -> str:
    return result_text.strip()

def save_incremental_results(batch_idx: int, results: Dict) -> str:
    """Save batch results incrementally to cache folder (results only)."""
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"a_generation_results_batch_{batch_idx}.json")
    with open(cache_path, "w", encoding="utf-8") as cache_file:
        json.dump(results, cache_file, ensure_ascii=False, indent=4)
    print(f"[INFO] Saved batch {batch_idx} to: {cache_path}")
    return cache_path

def load_incremental_results(start_batch: int = 1, end_batch: Optional[int] = None):
    """
    Load previously saved answer generation results from cache folder.
    
    Args:
        start_batch: Starting batch index (default: 1)
        end_batch: Ending batch index (default: None, loads all available)
    
    Returns:
        tuple: (merged_results_dict, last_batch_idx)
    """
    merged_results = {}
    last_batch_idx = start_batch - 1
    
    batch_idx = start_batch
    while True:
        if end_batch is not None and batch_idx > end_batch:
            break
            
        cache_path = os.path.join(CACHE_LOAD_FOLDER, f"a_generation_results_batch_{batch_idx}.json")
        
        if not os.path.exists(cache_path):
            if batch_idx == start_batch:
                print(f"[WARNING] No cache file found at batch {batch_idx}")
            break
        
        try:
            with open(cache_path, "r", encoding="utf-8") as cache_file:
                batch_results = json.load(cache_file)
            
            merged_results.update(batch_results)
            last_batch_idx = batch_idx
            
            print(f"[INFO] Loaded batch {batch_idx}: {len(batch_results)} results")
            batch_idx += 1
            
        except Exception as e:
            print(f"[ERROR] Failed to load batch {batch_idx}: {e}")
            break
    
    if last_batch_idx >= start_batch:
        print(f"\n[SUCCESS] Loaded batches {start_batch}-{last_batch_idx}")
        print(f"[INFO] Total merged results: {len(merged_results)}")
    else:
        print(f"[INFO] No batches loaded")
    
    return merged_results, last_batch_idx

In [ ]:
# Prepare answer inputs from question results
answer_inputs = []

for idx, item in enumerate(inputs):
    key = str(idx)
    questions = q_generation_results.get(key, [])
    context = item['text']

    for q_idx, question in enumerate(questions):
        answer_inputs.append({
            'article_idx': item['index'],
            'chunk_idx': item['chunk_idx'],
            'url': item['url'],
            'title': item['title'],
            'time': item['time'],
            'input_idx': idx,
            'q_idx': q_idx,
            'context': context,
            'question': question,
        })

print(f"Total answer inputs to process: {len(answer_inputs)}")

In [ ]:
# Define prompt formatter for answer generation
def format_answer_prompt(item: Dict) -> str:
    # Safely remove ID prefix - handle cases where question may not have the expected format
    question = item['question']
    if '-' in question:
        question = question.split('-', 1)[1]
    
    return A_INSTRUCTION.format(
        title=item['title'],
        timestamp=item['time'],
        text=item['context'],
        question=question
    )

# Create batch files
a_filenames = create_batch_files(
    inputs=answer_inputs,
    batch_path_template=BATCH_A_PATH,
    prompt_formatter=format_answer_prompt,
    gen_config=A_GEN_CONFIG
)


# ==================== INCREMENTAL BATCH PROCESSING ====================
if RESUME_FROM_LAST_CHECKPOINT:
    print(f"\n[INFO] Resuming from last checkpoint...")
    a_generation_results, last_batch = load_incremental_results()
    print(f"\n[INFO] Resume point: Last completed batch = {last_batch}")
else:
    a_generation_results = {}
    last_batch = 0


start_batch = last_batch + 1
a_errors = []

print(f"[INFO] Processing {len(a_filenames)} files...")
print(f"[INFO] Will start from batch {start_batch}")

for batch_idx, filename in enumerate(a_filenames, start=1):
    print(f"\n{'='*60}")
    print(f"[A-GEN] Processing batch {batch_idx}/{len(a_filenames)}: {filename}")
    print(f"{'='*60}")
    
    if batch_idx <= last_batch:
        print(f"[INFO] Skipping batch {batch_idx} as it is before resume point")
        continue

    # Check if batch already processed
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"a_generation_results_batch_{batch_idx}.json")
    if os.path.exists(cache_path):
        print(f"[INFO] Batch {batch_idx} already processed. Loading from cache...")
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                results = json.load(f)
            
            a_generation_results.update(results)
            
            print(f"[A-GEN] Batch {batch_idx}/{len(a_filenames)} loaded from cache")
            print(f"[A-GEN] Total results so far: {len(a_generation_results)}")
            continue
        except Exception as e:
            print(f"[WARNING] Failed to load cache for batch {batch_idx}: {e}")
            print(f"[INFO] Reprocessing batch {batch_idx}...")
    
    # Step 1: Submit batch job
    job = upload_and_submit_batch(filename)
    
    # Step 2: Wait for completion
    completed_job = wait_for_job_completion(job.name)
    
    # Step 3: Download and parse results
    results, errors = process_single_batch(
        filename=filename,
        job_name=job.name,
        parser=parse_answer
    )
    
    # Step 4: Merge results
    a_generation_results.update(results)
    a_errors.extend(errors)
    
    # Step 5: Save incrementally (results only)
    save_incremental_results(batch_idx, results)
    
    print(f"[A-GEN] Batch {batch_idx}/{len(a_filenames)} completed")
    print(f"[A-GEN] Total results so far: {len(a_generation_results)}")

print(f"\n{'='*60}")
print(f"[SUCCESS] All {len(a_filenames)} batch files processed!")
print(f"[INFO] Total results: {len(a_generation_results)}")
print(f"{'='*60}\n")

## 6. Save Results

Merge questions and answers into structured article format and save to Parquet.

In [ ]:
# Combine questions and answers, then save
# Structure: {article_idx: {'url', 'title', 'chunks': {chunk_idx: {context, qa_pairs: [{q, a}, ...]}}}}

article_data = {}

# First, build structure with questions
for idx, item in enumerate(inputs):
    key = str(idx)
    article_idx = item['index']
    chunk_idx = item['chunk_idx']
    questions = q_generation_results.get(key, [])

    if article_idx not in article_data:
        article_data[article_idx] = {
            'url': item['url'],
            'title': item['title'],
            'time': item['time'],
            'chunks': {}  # Use dict to handle non-sequential chunk indices
        }

    # Store chunk data by chunk_idx key
    if chunk_idx not in article_data[article_idx]['chunks']:
        article_data[article_idx]['chunks'][chunk_idx] = {'context': '', 'qa_pairs': []}

    article_data[article_idx]['chunks'][chunk_idx]['context'] = item['text']

    # Store questions with placeholders for answers
    for q in questions:
        article_data[article_idx]['chunks'][chunk_idx]['qa_pairs'].append({
            'question': q,
            'answer': ''
        })

# Now fill in answers
for idx, item in enumerate(answer_inputs):
    key = str(idx)
    answer = a_generation_results.get(key, '')
    article_idx = item['article_idx']
    chunk_idx = item['chunk_idx']
    q_idx = item['q_idx']

    if article_idx in article_data:
        if chunk_idx in article_data[article_idx]['chunks']:
            if q_idx < len(article_data[article_idx]['chunks'][chunk_idx]['qa_pairs']):
                article_data[article_idx]['chunks'][chunk_idx]['qa_pairs'][q_idx]['answer'] = answer

# Convert to output format - convert chunk dict to list (sorted by original chunk_idx)
outputs = []
for article_idx in sorted(article_data.keys()):
    data = article_data[article_idx]
    # Convert chunks dict to list, sorted by chunk_idx, preserving original index info
    chunks_list = [
        {'original_chunk_idx': k, **v} 
        for k, v in sorted(data['chunks'].items())
    ]
    outputs.append({
        'url': data['url'],
        'title': data['title'],
        'time': data['time'],
        'chunks': chunks_list
    })

df_output = pd.DataFrame(outputs)

# Save as parquet (handles nested structures well)
df_output.to_parquet(OUTPUT_QA_PARQUET, engine="pyarrow", index=False)
print(f"[INFO] Saved parquet: {OUTPUT_QA_PARQUET}")
print(f"[INFO] Total articles: {len(df_output)}")

### View Samples

Display sample QA pairs to verify output structure and quality.

In [ ]:
# Display sample QA pairs
num_samples = 3

print(f"[INFO] Sample structure")
print(f"Columns: {df_output.columns.tolist()}")

for idx in range(min(len(df_output), num_samples)):
    sample = df_output.iloc[idx]
    
    print(f"\n{'='*30}")
    print(f"SAMPLE #{idx + 1}")
    print(f"{'='*30}")
    
    url = str(sample['url'])
    title = str(sample['title'])
    print(f"url: {url[:50]}..." if len(url) > 50 else f"url: {url}")
    print(f"title: {title[:50]}..." if len(title) > 50 else f"title: {title}")
    print(f"chunks (count): {len(sample['chunks'])}")

    for i, chunk_data in enumerate(sample['chunks'][:2]):
        orig_idx = chunk_data.get('original_chunk_idx', i)
        print(f"  chunk_{i} (original_idx={orig_idx}):")
        print(f"    context: {chunk_data['context'][:400]}...")
        
        for j, qa in enumerate(chunk_data['qa_pairs'][:2]):
            print(f"    Q{j}: {qa['question']}")
            answer = qa['answer']
            print(f"    A{j}: {answer[:300]}..." if len(answer) > 300 else f"    A{j}: {answer}")